# Multi-Omic Data Integration Workflow

### Author: Kaili Fan

## Introduction

This notebook provides a reproducible workflow for integrating post-QC RNA and ATAC multi-omic data, specifically using a multi-tissue 10X Multiome dataset. The pipeline guides you through removing remaining doublets, checking cell overlap, transferring cell annotations from RNA to ATAC, identifying potential cis-regulatory elements, and performing missing data imputation. Given tissue specificity, data should be processed on a per-tissue basis. 

To follow this guide, you will need post-QC, cell-annotated RNA Anndata and post-QC ATAC Anndata objects as input.

In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join("..")) 
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

It is recommended to start the workflow from the 'multi-Omic_workflow/multiome_integration' directory. This ensures that all relative paths and function imports work as intended. Alternatively, you may use absolute paths if preferred.

In [2]:
os.getcwd()

'/data/fankaili/scripts/multi-Omic_workflow/multiome_integration/notebooks'

In [3]:
os.chdir('../')
os.getcwd()

'/data/fankaili/scripts/multi-Omic_workflow/multiome_integration'

All log outputs are stored in log files within the logs folder.

In [4]:
os.makedirs("test_output/logs", exist_ok=True)

## Config overview
### YAML File for Storing Input Parameters
All parameters and file paths used in this workflow are stored in config/joint_multiomic_config.yaml

Please review this configuration file to ensure that all parameters are correctly specified and as expected before proceeding with the analysis.

In [5]:
from multiome_integration.utils import load_config

config_path = "config/joint_multiomic_config.yaml"
config = load_config(config_path)
import yaml
print(yaml.dump(config, default_flow_style=False))

color:
  donor_colors:
    DGTEX001: '#397AF2'
    DGTEX002: '#E73A2F'
    DGTEX008: '#FBB30B'
    DGTEX027: '#8e02da'
    DGTEX031: '#65c4ff'
    DGTEX047: '#2E9E49'
  tissue_colors:
    Adipose - Subcutaneous: '#1d8c46'
    Adrenal Gland: '#33DD33'
    Kidney: '#a32015'
    Liver - Left Lobe: '#AABB66'
    Lung - Left Cranial Lobe: '#99FF00'
    Muscle - Gastrocnemius: '#9191ff'
    Muscle - Quadriceps: '#5e5eff'
    Thyroid: '#006600'
my_color_palette:
- '#c50909'
- '#ffa806'
- '#fbe95d'
- '#16b900'
- '#65c4ff'
- '#0e3aa9'
- '#8e02da'
- '#795548'
- '#9E9E9E'
- '#ff5b13'
- '#154f11'
- '#8bc34a'
- '#ff37a6'
- '#0062ff'
- '#00bfae'
- '#ea9c28'
- '#6e7f27'
- '#43e8e6'
- '#a4ff5a'
- '#e72c80'
params:
  celllineage_col: manual_broad_annotation
  celltype_col: manual_fine_annotation0
  imputation: true
  n_features: 50000
  n_threads: 16
  peak_half_width: 150
  suffix: Muscle_Quadriceps
  tissue: Muscle - Quadriceps
paths:
  atac_doublet_dir: /data/fankaili/scripts/multi-Omic_workflow/ATA

Make sure your files are formatted similarly, and update the file paths in config/joint_multiomic_config.yaml to point to your data files.

You can also modify the reference files in the configuration to match your dataset if needed.

- If you want to process all samples in sample_metatable, set tissue: --- in the config file; otherwise, specify a tissue to work on.
- 'suffix' is used for file naming. Please make sure the strings do not contain spaces or special characters.
- You can specify your own tissue or donor colors in the config file if desired; otherwise, the script will automatically assign colors.

### Table for Sample Information
All sample information is stored in the metatable, which is used throughout all sections of the workflow. You may use a comprehensive metatable containing all of your data; however, this workflow will process samples per designated tissue.

Please ensure your metatable follows the required format and includes all necessary sample data. Below is an example of how this table should look:

In [7]:
import pandas as pd

sample_metadata = pd.read_csv('../ATAC_QC/test_data/test_metatable.txt', sep='\t')

sample_metadata.head()

,DGTEX008-2BO-01-SM-OJX6G-EXP01-GEX-01,DGTEX008-2BO-01-SM-OJX6G-EXP01-ATAC-01,Human,DGTEX008,Toddler,M,Muscle - Quadriceps
0,DGTEX027-2BO-01-SM-OJX6J-EXP01-GEX-02,DGTEX027-2BO-01-SM-OJX6J-EXP01-ATAC-02,Human,DGTEX027,Pre-adult,M,Muscle - Quadriceps


## Step1 - check overlapping cells
Here, we will first remove remaining doublets by checking doublet cell barcodes detected in other other modality. If you do not want this step for some reason, just not provide the doublet file.

In [11]:
!python scripts/check_overlapping_cells.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/check_overlapping_cells.py config/joint_multiomic_config.yaml > logs/check_overlapping_cells.log 2>&1

Working tissue: Muscle - Quadriceps
RNA Samples: ['DGTEX008-2BO-01-SM-OJX6G-EXP01-GEX-01', 'DGTEX027-2BO-01-SM-OJX6J-EXP01-GEX-02']
ATAC Samples: ['DGTEX008-2BO-01-SM-OJX6G-EXP01-ATAC-01', 'DGTEX027-2BO-01-SM-OJX6J-EXP01-ATAC-02']
[INFO] ATAC doublet directory found: /data/fankaili/scripts/multi-Omic_workflow/ATAC_QC/test_output/doublet_filter_processing.demo_run/
[INFO] checking sample DGTEX027-2BO-01-SM-OJX6J-EXP01-GEX-02: Muscle_Quadriceps
[INFO] checking sample DGTEX008-2BO-01-SM-OJX6G-EXP01-GEX-01: Muscle_Quadriceps
Removing 120 remaining doublet cells for ATAC.
adata reduced from 4364 to 4317 cells.
[INFO] RNA doublet directory found: /data/fankaili/scripts/multi-Omic_workflow/RNA_QC/test_output/filter_qc.demo_run/
[INFO] checking sample DGTEX008-2BO-01-SM-OJX6G-EXP01-ATAC-01: Muscle - Quadriceps
[INFO] checking sample DGTEX027-2BO-01-SM-OJX6J-EXP01-ATAC-02: Muscle - Quadriceps
Removing 2083 remaining doublet cells for RNA.
adata reduced from 2977 to 2590 cells.
[INFO] Calculatin

## Step2 - ATAC cell annotation
In this step, we use GLUE (https://github.com/gao-lab/GLUE) to integrate post-QC RNA and ATAC data, thus transfer cell lables to all ATAC cells.

Note:

- This workflow uses some new functions in GLUE, which may not be available in the standard release. To ensure you have all the required features, please install GLUE using
      'pip install git+https://github.com/gao-lab/GLUE.git'

- If you have access to a GPU, this step can be significantly accelerated. Expected run time will be 15min-1hr, but could be longer if you have big dataset

In [12]:
!python scripts/annotate_ATAC_cells.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/annotate_ATAC_cells.py config/joint_multiomic_config.yaml > logs/annotate_ATAC_cells.log 2>&1

[INFO] torch.cuda.is_available(): True
[INFO] torch.cuda.device_count(): 8
[INFO] Using device: cuda

[INFO] Output path set to: /data/fankaili/scripts/multi-Omic_workflow/multiome_integration/test_output/glue

========== Processing tissue: Muscle - Quadriceps ==========
[INFO] Preparing RNA AnnData for 'Muscle_Quadriceps' using cell type column 'celltype_broad' ...
[INFO] Computing highly variable genes (HVGs) ...
Getting highly_variable
False    27903
True      4603
Name: count, dtype: int64 highly variable genes
[INFO] Plotting RNA UMAP before GLUE integration ...
Figure(640x480)
[INFO] Annotating genes using GTF: /data/fankaili/scripts/multi-Omic_workflow/ATAC_QC/ref/gencode.v47.primary_assembly.annotation.gtf ...
Retaining 32497 genes with usable positions.
... storing 'chrom' as categorical
... storing 'score' as categorical
... storing 'strand' as categorical
... storing 'thickStart' as categorical
... storing 'thickEnd' as categorical
... storing 'itemRgb' as categorical
... st

## Step3 - Identify cis-Regulatory Elements

Here, we call ATAC peaks to represent putative cis-regulatory elements. Peaks are called separately for each cell cluster within each tissue, then merged so that overlapping peaks are consolidated into single representative regions. 

By default, each peak is resized to 300 bp, but you can specify a different peak width in the config/joint_multiomic_config.yaml

In [13]:
!python scripts/identify_regulatory_elements.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/identify_regulatory_elements.py config/joint_multiomic_config.yaml > logs/identify_regulatory_elements.log 2>&1

[INFO] Output path set to: /data/fankaili/scripts/multi-Omic_workflow/multiome_integration/test_output/atac_peak
[INFO] Reading chromosome sizes from: /data/fankaili/ref/chromsize/hg38.chrom.sizes
[INFO] Loaded sizes for 455 chromosomes.

========== Processing tissue: Muscle - Quadriceps ==========
[INFO] Loading ATAC AnnData: /data/fankaili/scripts/multi-Omic_workflow/multiome_integration/test_output/h5ad/ATAC.GLUE.Muscle_Quadriceps.h5ad
[INFO] Calling peaks by cluster for tissue: Muscle_Quadriceps (n_jobs=16)
2025-07-25 10:25:09 - INFO - Exporting fragments...
[22s] ████████████████████████████████████████       3/3       (eta: 0s)        2025-07-25 10:25:38 - INFO - Calling peaks...
100%|███████████████████████████████████████████| 11/11 [03:39<00:00, 19.96s/it]
... storing 'celltype_std' as categorical
[INFO] Wrote ATAC with peaks: /data/fankaili/scripts/multi-Omic_workflow/multiome_integration/test_output/h5ad/ATAC_withPeak.Muscle_Quadriceps.h5ad
[INFO] Raw peaks for Vascular_Endo

## Step4 - Data imputation (Optional)

This step is optional.

In this step, we use MultiVI from the scvi-tools package to impute all signals, including missing values, for all cells. 

Please note that the imputed signals are dense and may not fully represent true biological measurements. You should carefully consider whether to perform imputation and which set of signals is most appropriate for your analysis.

In [14]:
!python scripts/data_imputation.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/data_imputation.py config/joint_multiomic_config.yaml > logs/data_imputation.log 2>&1

INFO: Seed set to 0
2025-07-25 10:30:06 - INFO - Seed set to 0
[INFO] torch.cuda.is_available(): True
[INFO] torch.cuda.device_count(): 8
[INFO] Using device: cuda

[INFO] Output path set to: /data/fankaili/scripts/multi-Omic_workflow/multiome_integration/test_output/multivi

========== Processing tissue: Muscle - Quadriceps ==========
[INFO] Loading RNA and ATAC AnnData for tissue: Muscle - Quadriceps
[INFO] Integrating all data...
[INFO] n(RNA): 4317, n(ATAC): 2590, n(shared): 1867, n(rna_only): 2450, n(atac_only): 723
[INFO] Assembling MultiVI input AnnData ...
[INFO] Extracting ATAC for 1867 cells
[INFO] Extracting RNA for 1867 cells (with celltype_broad)
[INFO] Extracting ATAC for 723 cells
[INFO] Extracting RNA for 2450 cells (with celltype_broad)
/home/fankaili/miniconda3/envs/workenv/lib/python3.12/site-packages/scvi/data/_preprocessing.py:327: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. 

## Step5 - Final integration

This step integrates the results from the previous steps and generates final h5ad files: one each for RNA-only, ATAC-only, and multiome cells. 

You can specify in the config/joint_multiomic_config.yaml whether you have imputed data. The imputed values will be stored in an additional layer alongside the raw counts.

In [15]:
!python scripts/data_integration.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/data_integration.py config/joint_multiomic_config.yaml > logs/data_integration.log 2>&1


========== Processing tissue: Muscle - Quadriceps ==========
[INFO] Loading AnnData for tissue: Muscle - Quadriceps
[INFO] Propagating RNA/ATAC obs fields ...
[INFO] Propagating sampleID
[INFO] Propagating leiden
[INFO] Propagating cellbarcode
[INFO] Propagating total_counts
[INFO] Propagating n_genes_by_counts
[INFO] Propagating n_genes
[INFO] Propagating pct_counts_mt
[INFO] Propagating pct_counts_ribo
[INFO] Propagating pct_counts_hb
[INFO] Propagating pct_exon_reads
[INFO] Propagating MALAT1_CPM
[INFO] Propagating doublet_score
[INFO] Propagating doublet_probabilities
[INFO] Propagating donorID
[INFO] Propagating sampleID
[INFO] Propagating leiden
[INFO] Propagating n_fragment
[INFO] Propagating frac_dup
[INFO] Propagating frac_mito
[INFO] Propagating tsse
[INFO] Adding var column (rna): gene_ids
[INFO] Adding var column (rna): feature_types
[INFO] Adding var column (rna): genome
[INFO] Adding var column (rna): mt
[INFO] Adding var column (rna): ribo
[INFO] Adding var column (rna)

## Bash Script to Run the Entire Multiomic Data Integrtaion Workflow
Below is the bash script notebooks/run_multiome_integration_workflow.sh that you can run the entire workflow. Alternatively, you may modify the script to run only specific steps as needed.

In [ ]:
# example bash command:
nohup bash notebooks/run_multiome_integration_workflow.sh > run_multiome_integration_workflow.log 2>&1 &